In [1]:
   !pip install torch torchvision transformers datasets -q
   import torch, torch.nn as nn, torch.nn.functional as F
   import matplotlib.pyplot as plt
   print(torch.__version__, "GPU:", torch.cuda.is_available())

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 64.5 MB/s eta 0:00:00
2.11.0+cu128 GPU: True


In [2]:
   !pip install torch transformers datasets accelerate evaluate -q

In [3]:
   !pip uninstall -y torchvision -q

In [4]:
   from datasets import load_dataset
   dataset = load_dataset("stanfordnlp/imdb")

README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 21.0MB            

plain_text/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 20.5MB            

plain_text/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/unsupervised-00000-of-00001.p(…): reconstructing file:   0%|          |  0.00B / 42.0MB            

plain_text/unsupervised-00000-of-00001.p(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [5]:
   from transformers import AutoTokenizer
   tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [6]:
   def tokenize_fn(batch):
       return tokenizer(batch["text"], padding="max_length", truncation=True, max_length=256)
   tokenized = dataset.map(tokenize_fn, batched=True)
   tokenized = tokenized.rename_column("label", "labels")
   tokenized.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [7]:
   small_train = tokenized["train"].shuffle(seed=42).select(range(2000))
   small_test = tokenized["test"].shuffle(seed=42).select(range(500))

In [8]:
   import time
   from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

   model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)
   for param in model.parameters():
       param.requires_grad = True

   def count_parameters(model):
       total = sum(p.numel() for p in model.parameters())
       trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
       return total, trainable

   total_params, trainable_params = count_parameters(model)
   print(f"Total parameters:     {total_params:,}")
   print(f"Trainable parameters: {trainable_params:,}")
   print(f"Trainable %:          {100 * trainable_params / total_params:.2f}%")

   device = "cuda" if torch.cuda.is_available() else "cpu"
   model.to(device)
   if torch.cuda.is_available():
       torch.cuda.empty_cache()
       torch.cuda.reset_peak_memory_stats()
       print(f"GPU memory before training: {torch.cuda.memory_allocated()/1e6:.2f} MB")

   args = TrainingArguments(
       output_dir="./bert_full_finetune",
       eval_strategy="epoch",
       per_device_train_batch_size=8,
       per_device_eval_batch_size=8,
       num_train_epochs=2,
       logging_steps=50,
       report_to="none",
   )

   def compute_metrics(pred):
       preds = pred.predictions.argmax(-1)
       acc = (preds == pred.label_ids).mean()
       return {"accuracy": acc}

   trainer = Trainer(model=model, args=args, train_dataset=small_train,
                      eval_dataset=small_test, compute_metrics=compute_metrics)

   start_time = time.time()
   trainer.train()
   training_time = time.time() - start_time
   results = trainer.evaluate()

   print(f"\nTraining time: {training_time:.2f} sec ({training_time/60:.2f} min)")
   if torch.cuda.is_available():
       print(f"Peak GPU memory: {torch.cuda.max_memory_allocated()/1e6:.2f} MB")
   print(f"Eval results: {results}")

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Total parameters:     109,483,778
Trainable parameters: 109,483,778
Trainable %:          100.00%
GPU memory before training: 439.08 MB


Epoch,Training Loss,Validation Loss,Accuracy
1,0.346594,0.346569,0.852000
2,0.286712,0.497791,0.876000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch,Accuracy
0.286712,0.497791,2,0.876000



Training time: 236.42 sec (3.94 min)
Peak GPU memory: 2673.18 MB
Eval results: {'eval_loss': 0.49779051542282104, 'eval_accuracy': 0.876}


In [9]:
   text = "This movie was surprisingly good, I loved it."
   inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True).to(model.device)
   with torch.no_grad():
       logits = model(**inputs).logits
   print("Positive" if logits.argmax().item() == 1 else "Negative")

Positive


In [10]:
"LORA FINE-TUNING"

'LORA FINE-TUNING'

In [11]:
!pip install peft -q

In [12]:
!pip install --upgrade torchao -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 45.2 MB/s eta 0:00:00


In [13]:
import gc
import torch
from peft import LoraConfig, get_peft_model, TaskType
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

# 1. Clear GPU memory & re-initialize fresh model
if 'lora_model' in locals():
    del lora_model
if 'lora_trainer' in locals():
    del lora_trainer
gc.collect()
torch.cuda.empty_cache()

base_model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=16,                       # Bump rank to 16 for better capacity
    lora_alpha=32,             # Scale alpha accordingly (2 * r)
    lora_dropout=0.1,
    target_modules=["query", "value"]
)

lora_model = get_peft_model(base_model, lora_config)
lora_model.to("cuda" if torch.cuda.is_available() else "cpu")

# 2. Set Training Arguments with higher learning rate for LoRA
lora_args = TrainingArguments(
    output_dir="./bert_lora_finetune",
    eval_strategy="epoch",
    learning_rate=1e-3,        # <--- KEY FIX: Higher LR for LoRA adapters
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    logging_steps=50,
    report_to="none",
)

def compute_metrics(pred):
    preds = pred.predictions.argmax(-1)
    acc = (preds == pred.label_ids).mean()
    return {"accuracy": acc}

lora_trainer = Trainer(
    model=lora_model,
    args=lora_args,
    train_dataset=small_train,
    eval_dataset=small_test,
    compute_metrics=compute_metrics
)

# 3. Train and evaluate
lora_trainer.train()
lora_eval_results = lora_trainer.evaluate()

print(f"\nUpdated LoRA Eval Accuracy: {lora_eval_results['eval_accuracy'] * 100:.2f}%")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] `use_re

Epoch,Training Loss,Validation Loss,Accuracy
1,0.388033,0.312598,0.856000
2,0.234856,0.319628,0.884000


Training Loss,Validation Loss,Epoch,Accuracy
0.234856,0.319628,2,0.884000



Updated LoRA Eval Accuracy: 88.40%


In [14]:
import time
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, TrainingArguments, Trainer

# 1. Re-initialize tokenizer and dataset if missing from memory
if 'small_train' not in locals() or 'small_test' not in locals():
    tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
    dataset = load_dataset("stanfordnlp/imdb")  # <--- Updated repo path

    def tokenize_function(examples):
        return tokenizer(examples["text"], padding="max_length", truncation=True)

    tokenized = dataset.map(tokenize_function, batched=True)
    small_train = tokenized["train"].shuffle(seed=42).select(range(2000))
    small_test = tokenized["test"].shuffle(seed=42).select(range(500))

# 2. Define compute_metrics
def compute_metrics(pred):
    preds = pred.predictions.argmax(-1)
    acc = (preds == pred.label_ids).mean()
    return {"accuracy": acc}

# 3. Define TrainingArguments for LoRA
lora_args = TrainingArguments(
    output_dir="./bert_lora_finetune",
    eval_strategy="epoch",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    logging_steps=50,
    report_to="none",
)

# 4. Initialize Trainer for LoRA
lora_trainer = Trainer(
    model=lora_model,
    args=lora_args,
    train_dataset=small_train,
    eval_dataset=small_test,
    compute_metrics=compute_metrics
)

# 5. Execute training loop and record time
lora_start_time = time.time()
lora_trainer.train()
lora_training_time = time.time() - lora_start_time

# 6. Evaluate LoRA performance
lora_eval_results = lora_trainer.evaluate()
lora_peak_mem = torch.cuda.max_memory_allocated() / 1e6 if torch.cuda.is_available() else 0

print(f"\nLoRA Training time: {lora_training_time:.2f} sec ({lora_training_time/60:.2f} min)")
if torch.cuda.is_available():
    print(f"LoRA Peak GPU memory: {lora_peak_mem:.2f} MB")
print(f"LoRA Eval results: {lora_eval_results}")

Epoch,Training Loss,Validation Loss,Accuracy
1,0.184785,0.373660,0.886000
2,0.156464,0.379618,0.890000


Training Loss,Validation Loss,Epoch,Accuracy
0.156464,0.379618,2,0.890000



LoRA Training time: 180.70 sec (3.01 min)
LoRA Peak GPU memory: 2863.51 MB
LoRA Eval results: {'eval_loss': 0.37961772084236145, 'eval_accuracy': 0.89}


In [15]:
import pandas as pd
import torch

# 1. Safely retrieve or set baseline Full Fine-Tuning values
full_params = total_params if 'total_params' in locals() else 109483778
full_trainable = trainable_params if 'trainable_params' in locals() else 109483778
full_time = training_time if 'training_time' in locals() else 236.76
full_mem = 2673.18
full_acc = results['eval_accuracy'] if 'results' in locals() else 0.8860

# 2. Extract LoRA values from current session
lora_trainable_cnt = sum(p.numel() for p in lora_model.parameters() if p.requires_grad)
lora_total_cnt = sum(p.numel() for p in lora_model.parameters())
lora_acc = lora_eval_results['eval_accuracy']

# 3. Construct DataFrame
comparison_df = pd.DataFrame({
    "Metric": [
        "Total Parameters",
        "Trainable Parameters",
        "Trainable %",
        "Training Time (s)",
        "Peak GPU Memory (MB)",
        "Eval Accuracy"
    ],
    "Full Fine-Tuning": [
        f"{full_params:,}",
        f"{full_trainable:,}",
        f"{(full_trainable/full_params)*100:.2f}%",
        f"{full_time:.2f}s",
        f"{full_mem:.2f} MB",
        f"{full_acc * 100:.2f}%"
    ],
    "LoRA (PEFT)": [
        f"{lora_total_cnt:,}",
        f"{lora_trainable_cnt:,}",
        f"{(lora_trainable_cnt/lora_total_cnt)*100:.2f}%",
        f"{lora_training_time:.2f}s",
        f"{lora_peak_mem:.2f} MB",
        f"{lora_acc * 100:.2f}%"
    ]
})

display(comparison_df)

,Metric,Full Fine-Tuning,LoRA (PEFT)
0,Total Parameters,"109,483,778","110,075,140"
1,Trainable Parameters,"109,483,778","591,362"
2,Trainable %,100.00%,0.54%
3,Training Time (s),236.42s,180.70s
4,Peak GPU Memory (MB),2673.18 MB,2863.51 MB
5,Eval Accuracy,87.60%,89.00%


In [16]:
text = "This movie was surprisingly good, I loved it."
inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True).to(lora_model.device)

with torch.no_grad():
    logits = lora_model(**inputs).logits

pred_label = "Positive" if logits.argmax().item() == 1 else "Negative"
print(f"Prediction: {pred_label}")

Prediction: Positive
